# RAG Streamlit Application

This notebook sets up a Retrieval-Augmented Generation (RAG) pipeline using LangChain, HuggingFace models, FAISS for vector storage, and deploys it as a Streamlit application via ngrok in Google Colab.

## 1. Environment Setup

Mount Google Drive to access your virtual environment and data files (e.g., `attention.pdf`).

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Install necessary Python packages. If you are using a virtual environment on Google Drive, you might install them there to persist across sessions.

In [ ]:
# Install virtual environment package (uncomment and run if setting up virtual env for the first time)
# !pip install virtualenv

# Create a virtual environment (uncomment and run if setting up virtual env for the first time)
# This path should match where you intend to store your virtual environment on Google Drive
# !virtualenv /content/drive/MyDrive/virtual_env

In [3]:
# Install packages into the virtual environment (uncomment and run if setting up virtual env for the first time)
# !source /content/drive/MyDrive/virtual_env/bin/activate; pip install langchain langchain-core langchain-community
# !source /content/drive/MyDrive/virtual_env/bin/activate; pip install chromadb torch transformers accelerate sentence-transformers openpyxl pacmap datasets

# Install the required packages for the current notebook session
!pip install pypdf bitsandbytes>=0.46.1 faiss-cpu langchain-huggingface streamlit pyngrok

Add the virtual environment's site-packages path to the system path, so Python can find the installed packages. Adjust the path if your Python version or virtual environment name is different.

In [4]:
import sys
import os

# Add the path of the virtual environment site-packages to colab system path
# Please verify the python version (e.g., python3.12) and the virtual env path
sys.path.append("/content/drive/MyDrive/virtual_env/lib/python3.12/site-packages")

## 2. Document Loading and Processing

Load documents from various sources: web and PDF. The `Transformer.txt` example is commented out as it was not included in the final `app.py`.

In [5]:
from langchain_community.document_loaders import TextLoader, WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Text loader example (currently not used in the RAG chain)
# Uncomment if want to try it
# loader_text = TextLoader("/content/drive/MyDrive/Colab Notebooks/RAG/Transformer.txt")
# text_docs = loader_text.load()

# Web based loader
loader_web = WebBaseLoader("https://www.ibm.com/think/topics/transformer-model/")
web_docs = loader_web.load()

# Pdf loader
# Ensure the path to your PDF is correct on Google Drive
loader_pdf = PyPDFLoader('/content/drive/MyDrive/Colab Notebooks/RAG/attention.pdf')
pdf_docs = loader_pdf.load()

# Merging docs from all sources (excluding text_docs as per previous decisions)
# Add 'text_docs' also if using them from above        # docs = text_docs + web_docs + pdf_docs
docs = web_docs + pdf_docs
print(f"Total documents loaded: {len(docs)}")

# Splitting documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunked_docs = text_splitter.split_documents(docs)
print(f"Total chunks created: {len(chunked_docs)}")

Total documents loaded: 16
Total chunks created: 90


## 3. Embedding Model and Vector Store

Create a FAISS vector database using `HuggingFaceEmbeddings` with the `all-MiniLM-L6-v2` model for efficiency. Then set up a retriever to fetch relevant documents.

In [6]:
# Using a lightweight embedding model
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')

# Create FAISS vector store from document chunks
db = FAISS.from_documents(chunked_docs, embeddings)

# Setting up retriever to return the top 4 relevant documents
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 4}  # Retrieves top 4 similar results
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4. Large Language Model (LLM) and RAG Chain

Load a quantized smaller LLM (`TinyLlama/TinyLlama-1.1B-Chat-v1.0`) for faster inference and define the RAG chain.

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Model selection and quantization configuration
model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Load quantized model and tokenizer
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Create text generation pipeline
text_generation_pipeline = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    temperature=0.2,
    do_sample=True,
    repetition_penalty=1.1,
    return_full_text=False, # Important: Set to False to only return generated text
    max_new_tokens=400,
)

# Initialize HuggingFacePipeline for LangChain
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# Define ChatPromptTemplate for the RAG chain
prompt = ChatPromptTemplate.from_template("""
<|system|>
Answer the question based on your knowledge. Use the following context to help:

{context}

</s>
<|user|>
{question}
</s>
<|assistant|>""")

# Create the LLM chain
llm_chain = prompt | llm | StrOutputParser()

# Combine with the retriever to create the full RAG chain
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | llm_chain
)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_7687/108371565.py:34: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=text_generation_pipeline)


### Testing LLM and RAG without streamlit app first. Analyse the difference in their outputs.

In [8]:
# user query
question = "what is attention?"

In [9]:
# No context added, lets see the output
llm_chain.invoke({"context":"", "question": question})

Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\nAttention is a human cognitive process that involves the selective perception and processing of information in order to focus on a particular object or task. It is a vital component of human behavior, as it enables us to make decisions, solve problems, and interact with the world around us. Attention can be divided into two main types: automatic and controlled. Automatic attention is the unconscious process by which we respond to stimuli without conscious awareness. This type of attention is essential for everyday activities such as walking, eating, and talking. Controlled attention, on the other hand, is the conscious effort to focus our attention on a specific object or task. This type of attention allows us to make more informed decisions and to engage in more complex tasks. Overall, attention plays a crucial role in shaping our experiences and enabling us to navigate the world around us.'

In [10]:
# Output after adding context
rag_chain.invoke(question)

Both `max_new_tokens` (=400) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'\nAttention is a concept in artificial intelligence and natural language processing (NLP). It refers to the process of identifying and focusing on the most important parts of a given text or conversation. Attention can be used to improve the quality and efficiency of NLP systems by enabling them to better understand and analyze human language.\n\nIn NLP, attention is typically measured using a technique called the "attention weight matrix" (AWM), which involves calculating the probability of each token being present in a given sentence or document. This probability is then used to weight the tokens according to their importance to the overall message. By focusing on the most important tokens, NLP systems can improve their ability to understand and interpret human language more accurately.\n\nFor example, in the context of natural language processing, a system might use attention to identify the most important words or phrases in a sentence, such as "the quick brown fox jumps over the 

## 5. Streamlit App Definition (`app.py`)

Write the Streamlit application code to a file named `app.py`. This script will be executed by Streamlit.

In [11]:
%%writefile app.py
import streamlit as st
import torch
import sys
import os # Import os for path checking

# Ensure Google Drive is mounted for virtual environment and data access
# This script assumes /content/drive is already mounted from a Colab session.
# If running standalone, you might need to handle mounting interactively or ensure paths are accessible.

# Add the path of the virtual environment site-packages to colab system path
# This assumes the virtual_env is created and packages are installed as per the notebook.
# Please ensure python3.12 matches your virtual environment's python version.
# If your virtual environment is not 'python3.12', adjust the path accordingly.
sys.path.append("/content/drive/MyDrive/virtual_env/lib/python3.12/site-packages")

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from langchain_community.document_loaders import TextLoader, WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import HuggingFacePipeline
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


@st.cache_resource
def setup_rag_pipeline():
    # 1. Load Documents
    # Ensure these paths are correct and accessible from the mounted drive
    # loader_text = TextLoader("/content/drive/MyDrive/Colab Notebooks/RAG/Transformer.txt")
    # text_docs = loader_text.load()

    loader_web = WebBaseLoader("https://www.ibm.com/think/topics/transformer-model/")
    web_docs = loader_web.load()

    loader_pdf = PyPDFLoader('/content/drive/MyDrive/Colab Notebooks/RAG/attention.pdf')
    pdf_docs = loader_pdf.load()

    # Merging docs from all sources
    docs = web_docs + pdf_docs

    # 2. Split Documents
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunked_docs = text_splitter.split_documents(docs)

    # 3. Create Vector Store (FAISS)
    embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
    db = FAISS.from_documents(chunked_docs, embeddings)
    retriever = db.as_retriever(search_type="similarity", search_kwargs={'k': 4})

    # 4. Load LLM
    model_name = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=bnb_config)
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    text_generation_pipeline = pipeline(
        model=model,
        tokenizer=tokenizer,
        task="text-generation",
        temperature=0.2,
        do_sample=True,
        repetition_penalty=1.1,
        return_full_text=False, # Set to False as discussed to avoid returning the full prompt
        max_new_tokens=400,
    )
    llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

    # 5. Define Prompt Template
    prompt_template = """<|system|>
Answer the question based on your knowledge. Use the following context to help:

{context}

</s>
<|user|>
{question}
</s>
<|assistant|>"""
    prompt = ChatPromptTemplate.from_template(prompt_template)

    # 6. Create RAG Chain
    llm_chain = prompt | llm | StrOutputParser()
    rag_chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | llm_chain
    )
    return rag_chain

# Streamlit UI
st.title('Transformer RAG Demo')

# Setup RAG pipeline (cached)
rag_chain = setup_rag_pipeline()

input_text = st.text_input("Ask a question about Transformers:")

if input_text:
    with st.spinner("Searching and generating response..."):
        response = rag_chain.invoke(input_text)
        st.write(response)

Writing app.py


## 6. Run Streamlit App with `ngrok`

Use `ngrok` to create a public URL for your Streamlit application running on Colab. You need to provide your `ngrok` authtoken, which should be stored in Colab secrets as `NGROK__AUTH_TOKEN`.

In [12]:
from pyngrok import ngrok
from google.colab import userdata
import time

# Terminate any previous ngrok tunnels to avoid conflicts
ngrok.kill()

# Fetch ngrok authtoken from Colab secrets
# Ensure you have added your NGROK_AUTHTOKEN to Colab secrets
NGROK_AUTHTOKEN = userdata.get('NGROK__AUTH_TOKEN') ## provide the same token name here as the colab ngrok secret key
ngrok.set_auth_token(NGROK_AUTHTOKEN)

# Run Streamlit in the background and redirect its output to a log file for debugging
# Streamlit typically runs on port 8501
!nohup streamlit run app.py --server.port 8501 &> streamlit_app.log &

# Give Streamlit a moment to start up
time.sleep(5)

# Create a public URL for the Streamlit app
public_url = ngrok.connect(addr='8501')
print(f"Your Streamlit app is live at: {public_url}")

# You can check the logs if the app doesn't appear to be running
# !cat streamlit_app.log

Your Streamlit app is live at: NgrokTunnel: "https://katherine-protrusile-shamefully.ngrok-free.dev" -> "http://localhost:8501"
